In [21]:
import re
import os
import arxiv
import asyncio
import requests
from langchain.tools import tool
from langchain_groq import ChatGroq
from semanticscholar import SemanticScholar
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from typing import TypedDict, List,Optional, Annotated,Sequence,Set
from langchain_core.messages import BaseMessage,SystemMessage,HumanMessage

In [28]:
def sanitize_filename(name: str) -> str:
    return re.sub(r'[\\/*?:"<>|]', "_", name)

async def arxiv_searcher(arxiv_phrases: List[str]) -> Optional[Set[str]]:
    """
    Return the URLs of 5 papers for each keyword from arxiv, and download PDFs.
    """
    results = set()

    for phrase in arxiv_phrases:
        search = arxiv.Search(query=phrase, max_results=5)
        for paper in search.results():
            results.add(paper.pdf_url)
            
            safe_title = sanitize_filename(paper.title)
            await asyncio.to_thread(
                paper.download_pdf,
                dirpath=f"./papers/{safe_title}"
            )

    return results

In [ ]:
def download_pdf(url: str, filepath: str):
    resp = requests.get(url)
    resp.raise_for_status()
    with open(filepath, "wb") as f:
        f.write(resp.content)

def save_abstract(text: str, filepath: str):
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)

async def semantic_scholar_searcher(keywords: List[str]) -> Optional[Set[str]]:
    """
    For each keyword:
    - If open access PDF exists → download PDF.
    - Else → save abstract as .txt file.
    Returns the set of file paths saved.
    """
    client = SemanticScholar()
    saved_files: Set[str] = set()
    os.makedirs("./papers", exist_ok=True)

    for kw in keywords:
        response = await asyncio.to_thread(
            client.search_paper,
            query=kw,
            limit=5
        )

        for paper in response:
            pdf_info = getattr(paper, 'openAccessPdf', None)
            title = sanitize_filename(paper.title or paper.paperId)

            if pdf_info and pdf_info.get('url'):
                pdf_path = f"./papers/{title}.pdf"
                await asyncio.to_thread(download_pdf, pdf_info['url'], pdf_path)
                saved_files.add(pdf_path)
            else:
                abstract_text = paper.abstract or "[No abstract available]"
                txt_path = f"./papers/{title}.txt"
                save_abstract(abstract_text, txt_path)
                saved_files.add(txt_path)

    return saved_files

In [24]:
class Clarification(TypedDict):
    needs_improvement :  bool
    questions : Optional[List[str]]

class KeywordExtractionOutput(TypedDict):
    arxiv_phrases: List[str]
    semantic_scholar_queries: List[str]


class AgentState(TypedDict):
    clarification : Clarification
    keywords : KeywordExtractionOutput
    messages : Annotated[Sequence[BaseMessage],add_messages]

In [25]:
clarifier_llm = ChatGroq(model="moonshotai/kimi-k2-instruct").with_structured_output(Clarification)
keyword_llm = ChatGroq(model="llama3-70b-8192").with_structured_output(KeywordExtractionOutput)

In [26]:
def clarifier(state:AgentState):
    system_prompt = SystemMessage(content="""
        You are a research assistant. The user will provide you with a research paper topic or description.
        
        Your job is to check if the description includes:
        1. Research domain
        2. Problem being solved
        3. Method or technique used
        4. Any dataset mentioned
        
        If ANY of these elements are missing or unclear, set needs_improvement to True and provide specific questions in the 'questions' field to gather the missing information.
        
        If all elements are present and clear, set needs_improvement to False and questions can be null or empty.
        
        Example response format:
        - If missing info: {"needs_improvement": true, "questions": ["What specific problem are you trying to solve?", "Which dataset will you use?"]}
        - If complete: {"needs_improvement": false, "questions": null}
        """
    )
    clarifier_response = clarifier_llm.invoke([system_prompt]+[state["messages"][-1]])

    return {"clarification":clarifier_response}

In [27]:
def keyworder(state:AgentState):
    system_prompt = SystemMessage(content="""
        You are a research assistant trained to extract keywords for academic paper search engines, specifically for:
        - arXiv (https://arxiv.org)
        - Semantic Scholar (https://semanticscholar.org)

        You will receive a short research topic description from the user.

        Your task is to analyze the description and return a set of structured keywords and search phrases optimized for academic search.

        You MUST return your output as an instance of the following schema:

        class KeywordExtractionOutput(BaseModel):
            arxiv_phrases: List[str]  # short, concise phrases optimized for arXiv title/abstract search
            semantic_scholar_queries: List[str]  # natural language-style search strings for Semantic Scholar

        Guidelines:
        - Avoid generic terms like "paper", "study", "research"
        - Prefer specific methods (e.g., CNN, BERT, PCA), tasks (e.g., segmentation, prediction), and datasets (e.g., CHB-MIT, ImageNet)
        - If user input is vague, extract the most relevant, inferable terms — dont leave the lists empty
        - All outputs should be lowercase unless referring to acronyms (e.g., EEG, GNN, LSTM)

        Only return a valid Python object matching the schema exactly.
        Do not include any extra fields, strings, comments, or explanations.
        Avoid quoting the entire object as a string.
        """
        )
    
    keyworder_response = keyword_llm.invoke([system_prompt] + state['messages'])

    return {"keywords":keyworder_response}
    

In [7]:
def clarifier_router(state: AgentState):
    if state['clarification']['needs_improvement']:
        return "end"
    else:
        return "continue"

In [8]:
graph = StateGraph(AgentState)

graph.add_node("clarificationAgent",clarifier)
graph.add_node("keywordAgent",keyworder)

graph.add_edge(START,"clarificationAgent")

graph.add_conditional_edges(
    "clarificationAgent",
    clarifier_router,
    {
        "end":END,
        "continue": "keywordAgent"
    }
)

graph.add_edge("keywordAgent",END)

app = graph.compile()

In [13]:
user_input_messages = [HumanMessage(content="""
The title of my paper is preictal state recognition using geometric deep learning. Im trying to improve the early detection of preictal (pre-seizure) brain states in patients with epilepsy using EEG data. 
                                    The goal is to predict seizure onset several minutes in advance so preventive interventions can be applied, especially in wearable or edge devices. 
                                    I plan to use a Graph Neural Network (GNN) architecture, specifically a spatio-temporal GCN, to model both the spatial brain connectivity and the temporal patterns leading up to a seizure
                                    .Ill be using the CHB-MIT Scalp EEG dataset, 
                                    which contains long-term EEG recordings from pediatric subjects with intractable seizures, 
                                    including annotations for seizure onset and preictal windows.
""")]

# Initialize state properly
state = {
    "messages": user_input_messages, 
    "clarification": {"needs_improvement": False, "question": None}, 
    "keywords": ""
}

for event in app.stream(state):
    for node_name, node_output in event.items():
        print(f"\n🧩 Agent: {node_name}")
        print(f"📦 Output: {node_output}")
        state.update(node_output)
        
    if node_name == END:
        break
        
    if state.get('clarification', {}).get('needs_improvement', False):
        message = input("Answer: ")
        if message.lower() == "exit":
            break
        state["messages"].append(HumanMessage(content=message))



🧩 Agent: clarificationAgent
📦 Output: {'clarification': {'needs_improvement': False, 'questions': []}}

🧩 Agent: keywordAgent
📦 Output: {'keywords': {'arxiv_phrases': ['preictal state detection', 'eeg-based seizure prediction', 'geometric deep learning for epilepsy', 'spatio-temporal gcn for eeg analysis'], 'main_keywords': ['geometric deep learning', 'preictal state recognition', 'graph neural network', 'spatio-temporal gcn', 'eeg', 'seizure prediction', 'chb-mit'], 'semantic_scholar_queries': ['early detection of preictal states in epilepsy using eeg', 'graph neural networks for seizure prediction', 'chb-mit dataset for eeg analysis', 'geometric deep learning for brain connectivity modeling']}}
